In [3]:
import cv2
import numpy as np
import os

def estimate_dominant_intensity_range(image, lower_percentile=10, upper_percentile=90):
    """
    Estimate dominant SEM intensity band using percentiles.
    """
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    low = np.percentile(gray, lower_percentile)
    high = np.percentile(gray, upper_percentile)
    return low, high


def augment_image(input_dir, output_dir, blur_ksize, noise_sigma):
    """
    Apply Gaussian blur and intensity-aware Gaussian noise
    to all images in a directory.
    """

    os.makedirs(output_dir, exist_ok=True)

    for fname in os.listdir(input_dir):

        image_path = os.path.join(input_dir, fname)
        image = cv2.imread(image_path)

        if image is None:
            continue

        # ---------------------------------
        # Estimate dominant intensity band
        # ---------------------------------
        low_int, high_int = estimate_dominant_intensity_range(image)

        # ---------------------------------
        # Apply Gaussian blur
        # ---------------------------------
        if blur_ksize % 2 == 0:
            blur_ksize += 1

        blurred = cv2.GaussianBlur(image, (blur_ksize, blur_ksize), 0)

        # ---------------------------------
        # Generate grayscale Gaussian noise
        # ---------------------------------
        noise_gray = np.random.normal(
            0, noise_sigma,
            (image.shape[0], image.shape[1], 1)
        ).astype(np.float32)

        noise = np.repeat(noise_gray, 3, axis=2)

        # ---------------------------------
        # Apply noise ONLY in dominant band
        # ---------------------------------
        gray_blurred = cv2.cvtColor(blurred, cv2.COLOR_BGR2GRAY)
        mask = (gray_blurred >= low_int) & (gray_blurred <= high_int)

        noisy_blurred = blurred.astype(np.float32)

        # Apply noise only where mask is True
        noisy_blurred[mask] += noise[mask]

        # ---------------------------------
        # Clip + convert
        # ---------------------------------
        noisy_blurred = np.clip(noisy_blurred, 0, 255).astype(np.uint8)

        output_path = os.path.join(output_dir, fname)
        cv2.imwrite(output_path, noisy_blurred)

        print(f"Saved: {fname}")


# Example usage
if __name__ == "__main__":
    augment_image(
        "cropped",
        "blurred",
        blur_ksize=17,
        noise_sigma=35
    )

Saved: cropped_prac7_etched_000.tif
Saved: cropped_prac7_etched_001.tif
Saved: cropped_prac7_etched_002.tif
Saved: cropped_prac7_etched_003.tif
Saved: cropped_prac7_etched_004.tif
Saved: cropped_prac7_etched_005.tif
Saved: cropped_prac7_etched_006.tif
Saved: cropped_prac7_etched_007.tif
Saved: cropped_prac7_etched_008.tif
Saved: cropped_prac7_etched_009.tif
Saved: cropped_prac7_etched_010.tif
Saved: cropped_prac7_etched_011.tif
Saved: cropped_prac7_etched_012.tif
Saved: cropped_prac7_etched_013.tif
Saved: cropped_prac7_etched_014.tif
Saved: cropped_prac7_etched_015.tif
Saved: cropped_prac7_etched_016.tif
Saved: cropped_prac7_etched_017.tif
Saved: cropped_prac7_etched_018.tif
Saved: cropped_prac7_etched_019.tif
Saved: cropped_prac7_etched_020.tif
Saved: cropped_prac7_etched_021.tif
Saved: cropped_prac7_etched_022.tif
Saved: cropped_prac7_etched_023.tif
Saved: cropped_prac7_etched_024.tif
Saved: cropped_prac7_etched_025.tif
Saved: cropped_prac7_etched_026.tif
Saved: cropped_prac7_etched_